# Session 4: Convolutional Neural Networks and Filters

This notebook contains structured tasks based on Session 4 materials.

**Instructions:**
- Do NOT use a separate `.py` file. Complete all tasks directly in this notebook.
- Keep your code clear and well-commented.
- For Visulaization, please use `%matplotlib inline`
- Submit your completed notebook by uploading it to your forked repository.

### ***DO NOT UPLOAD THE DATASET TO GITHUB as it's too large***
<hr>

## Task 1: Convolutional Filters on an Image
Apply several 2D filters (Sobel X, Sobel Y, Laplacian, and Sharpen) to an image and visualize the results side by side.

**Hint:**
- Use OpenCV (`cv2.filter2D`) or NumPy to apply kernels.
- Define common kernels (Sobel, Laplacian, Sharpen).
- Use `matplotlib.pyplot` to plot the original and filtered images.

In [ ]:
%matplotlib inline
# TODO: Load an image (use OpenCV or matplotlib)
# TODO: Convert to grayscale
# TODO: Define Sobel, Laplacian, and Sharpen kernels
# TODO: Apply each filter to the image
# TODO: Plot results in a grid using matplotlib
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

path = r"D:\SkyXperts-Vision-Course\Session4\Feature_Vectors\images\jeep1.jpeg"

# Load the image (OpenCV loads color images in BGR order)
color_img = cv.imread(path)
if color_img is None:
    raise FileNotFoundError(f"Could not read image at: {path}")

# Convert to grayscale (filters work on a single intensity channel)
img = cv.cvtColor(color_img, cv.COLOR_BGR2GRAY)

# Define the kernels myself
# Sobel X: responds to horizontal intensity change -> finds vertical edges
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]])
# Sobel Y: responds to vertical intensity change -> finds horizontal edges
sobel_y = np.array([[ 1,  2,  1],
                    [ 0,  0,  0],
                    [-1, -2, -1]])
# Laplacian: second derivative -> edges in all directions
laplacian = np.array([[0,  1, 0],
                      [1, -4, 1],
                      [0,  1, 0]])
# Sharpen: original image + edge boost
sharpen = np.array([[ 0, -1,  0],
                    [-1,  5, -1],
                    [ 0, -1,  0]])

def apply_kernel(image, kernel):
    # CV_64F keeps negative values (uint8 would clip them to 0);
    # convertScaleAbs then takes |value| and converts back to uint8 for display
    filtered = cv.filter2D(image, cv.CV_64F, kernel)
    return cv.convertScaleAbs(filtered)

# Apply each filter to the image
sobeled_x   = apply_kernel(img, sobel_x)
sobeled_y   = apply_kernel(img, sobel_y)
laplacianed = apply_kernel(img, laplacian)
# Sharpen should stay a normal-looking image, so clip to 0-255 instead of abs
sharpened   = np.clip(cv.filter2D(img, cv.CV_64F, sharpen), 0, 255).astype(np.uint8)

# Plot results in a 2x3 grid
results = [
    ("Original Image", img),
    ("Sobel X", sobeled_x),
    ("Sobel Y", sobeled_y),
    ("Laplacian", laplacianed),
    ("Sharpened", sharpened),
]
plt.figure(figsize=(10, 8))
for i, (title, im) in enumerate(results, start=1):
    plt.subplot(2, 3, i)
    plt.imshow(im, cmap='gray')
    plt.title(title)
    plt.axis('off')
plt.tight_layout()
plt.show()


## Task 2: Build a Simple CNN on FashionMNIST
Construct and train a small Convolutional Neural Network (CNN) on the FashionMNIST dataset.

**Hint:**
- Use `torchvision.datasets.FashionMNIST` for loading data.
- Define a small CNN with `nn.Conv2d`, `nn.ReLU`, `nn.MaxPool2d`, and `nn.Linear`.
- Train for a few epochs using an optimizer (e.g., Adam) and loss function (e.g., CrossEntropyLoss).
- Print training accuracy after each epoch.

In [ ]:
# TODO: Import torch, torchvision, and related libraries
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim

# TODO: Load FashionMNIST dataset with transforms
# ToTensor: PIL image (28x28, 0-255) -> float tensor (1x28x28, 0-1)
transform = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])
trainset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True)

# TODO: Define a simple CNN class with one conv layer and one fc layer
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 32 learned 3x3 filters; padding=1 keeps the size at 28x28
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()                 # nonlinearity (without it, conv+fc collapse into one linear map)
        self.pool = nn.MaxPool2d(2)           # 28x28 -> 14x14
        self.fc1 = nn.Linear(32 * 14 * 14, 10)  # 10 clothing classes

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))  # (B, 32, 14, 14)
        x = x.view(x.size(0), -1)                # flatten to (B, 6272)
        x = self.fc1(x)                          # raw scores (logits)
        return x

# TODO: Set up optimizer and loss function
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()                # expects raw logits + integer labels
optimizer = optim.Adam(model.parameters(), lr=0.001)

# TODO: Implement a short training loop (2-3 epochs)
for epoch in range(3):
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in trainloader:
        optimizer.zero_grad()                    # clear old gradients
        outputs = model(inputs)                  # forward pass
        loss = criterion(outputs, labels)        # how wrong are we?
        loss.backward()                          # compute gradients
        optimizer.step()                         # update weights

        running_loss += loss.item()
        predicted = outputs.argmax(dim=1)        # class with the highest score
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    # Print training accuracy after each epoch
    print(f'Epoch {epoch + 1}, Loss: {running_loss / len(trainloader):.4f}, '
          f'Train Accuracy: {100 * correct / total:.2f}%')


## Task 3: Visualize Feature Maps
Visualize feature maps produced by the first convolutional layer of your CNN.

**Hint:**
- Take a single image from the test set.
- Pass it through the first convolutional layer (`net.conv1`).
- Convert the output to NumPy and plot several channels as images.

In [ ]:
# TODO: Take one test image
# TODO: Forward pass through the first convolutional layer
# TODO: Convert feature maps to NumPy array
# TODO: Plot the first few feature maps in a grid
model.eval()
with torch.no_grad():  # no gradients needed -> less memory, and .numpy() works directly
    testset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=1, shuffle=False)
    dataiter = iter(testloader)
    images, labels = next(dataiter)          # one test image: shape (1, 1, 28, 28)

    feature_maps = model.conv1(images)       # first conv layer only: shape (1, 32, 28, 28)
    feature_maps_np = feature_maps.numpy()   # convert to NumPy

    # Plot the original image, then the first 7 feature maps (2x4 grid)
    plt.figure(figsize=(12, 6))
    plt.subplot(2, 4, 1)
    plt.imshow(images[0, 0].numpy(), cmap='gray')
    plt.title('Original')
    plt.axis('off')
    for i in range(7):
        plt.subplot(2, 4, i + 2)
        plt.imshow(feature_maps_np[0, i], cmap='gray')   # image 0, filter i
        plt.title(f'Feature Map {i + 1}')
        plt.axis('off')
    plt.tight_layout()
    plt.show()


## Task 4: Haar Cascade Face Detection

Apply Haar Cascade classifiers for basic object detection (faces).

**Hint:**
- Use `cv2.CascadeClassifier` with a pre-trained XML file (e.g., `haarcascade_frontalface_default.xml`).
- Convert the input image to grayscale before detection.
- Use `detectMultiScale` to get bounding boxes.
- Draw rectangles on detected faces with `cv2.rectangle`.
- Visualize with Matplotlib.


In [ ]:
# OpenCV 5 moved CascadeClassifier to the contrib package, so stay on OpenCV 4.
# After running this cell: Kernel -> Restart, then run the Task 4 cell below.
%pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless
%pip install "opencv-python<5"


In [ ]:
# TODO: Load an image containing a face
# TODO: Convert to grayscale
# TODO: Load Haar Cascade XML (frontal face) || cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
# TODO: Apply detectMultiScale to find faces
# TODO: Draw bounding boxes on the original image
# TODO: Display results with matplotlib
import cv2 as cv
import matplotlib.pyplot as plt

path = r"D:\SkyXperts-Vision-Course\Session4\Feature_Vectors\images\face1.jpg"

# Load the color image (this is the "original" we will draw on)
image = cv.imread(path)
if image is None:
    raise FileNotFoundError(f"Could not read image at: {path}")

# Grayscale copy used only for detection
gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)

# Load Haar cascade XML (frontal face)
face_cascade = cv.CascadeClassifier(cv.data.haarcascades + 'haarcascade_frontalface_default.xml')
if face_cascade.empty():
    raise RuntimeError("Unable to load Haar cascade classifier.")

# Detect faces: scaleFactor = shrink step per pass, minNeighbors = agreement needed
faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
print(f"Faces found: {len(faces)}")

# Draw bounding boxes on the original (color) image; (255, 0, 0) is blue in BGR
for (x, y, w, h) in faces:
    cv.rectangle(image, (x, y), (x + w, y + h), (255, 0, 0), 2)

# Display with matplotlib (convert BGR -> RGB first)
plt.imshow(cv.cvtColor(image, cv.COLOR_BGR2RGB))
plt.axis('off')
plt.show()
